In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file for March 9, 2021
metric_container_path = 'dataset/Bank/telemetry/2021_03_09/metric/metric_container.csv'
metric_container_df = pd.read_csv(metric_container_path)

# Extract unique KPI names and their corresponding components (cmdb_id)
kpi_component_mapping = metric_container_df[['cmdb_id', 'kpi_name']].drop_duplicates()

kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file for March 9, 2021, contains 1689 unique KPI names associated with various components. These components include Tomcat04, MG01, Tomcat02, and others. Each KPI name represents a specific metric, such as CPU utilization, memory usage, process counts, session details, and thread information. This data provides a comprehensive mapping of KPIs to components, which can be used to identify relevant metrics for diagnosing root causes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name
0       Tomcat04                         OSLinux-CPU_CPU_CPUCpuUtil
1           MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies
2           MG01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem
3           MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount
4           MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
...          ...                                                ...
520542  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...
520543  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
520544  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
520545  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...
520547  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...

[1689 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Group data by 'component-KPI' (cmdb_id and kpi_name) and aggregate time series
metric_container_df['component_KPI'] = metric_container_df['cmdb_id'] + '-' + metric_container_df['kpi_name']
component_kpi_series = metric_container_df.groupby('component_KPI')['value']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = component_kpi_series.quantile(0.95).reset_index()
p95_thresholds.columns = ['component_KPI', 'P95_threshold']

p95_thresholds

```
Out[2]:
```
The global P95 thresholds for each 'component-KPI' time series in the `metric_container.csv` file for March 9, 2021, have been calculated. There are 1689 unique 'component-KPI' combinations, each with its corresponding P95 threshold value. These thresholds represent the 95th percentile of the metric values for the entire day's data, providing a benchmark for identifying abnormal behavior in the system components.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI  P95_threshold
0         IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1       IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.139625e+01
2        IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.476656e+08
3      IG01-JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.795390e+07
4        IG01-JVM-Operating System_7778_JVM_JVM_CPULoad   5.109200e-01
...                                                 ...            ...
1684  dockerB2-Container-DOCKER_CONTAINER_b2fc383d24...   0.000000e+00
1685  dockerB2-Container-DOCKER_CONTAINER_b2fc383d24...   0.000000e+00
1686  dockerB2-Container-DOCKER_CONTAINER_b2fc383d24...   0.000000e+00
1687  dockerB2-Container-DOCKER_CONTAINER_b2fc383d24...   0.000000e+00
1688  dockerB2-Container-DOCKER_CONTAINER_b2fc383d24...   0.000000e+00

[1689 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert timestamps to UTC+8 and filter data within the specified time range
utc8 = pytz.timezone('Asia/Shanghai')
metric_container_df['timestamp_utc8'] = pd.to_datetime(metric_container_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert(utc8)

start_time = utc8.localize(datetime(2021, 3, 9, 16, 30))
end_time = utc8.localize(datetime(2021, 3, 9, 17, 0))

filtered_data = metric_container_df[(metric_container_df['timestamp_utc8'] >= start_time) & (metric_container_df['timestamp_utc8'] < end_time)]

filtered_data

```
Out[3]:
```
The data within the specified time range of March 9, 2021, from 16:30 to 17:00 (UTC+8) has been successfully filtered from the `metric_container.csv` file. A total of 25,939 rows of data were extracted, representing various 'component-KPI' time series during this period. This filtered dataset can now be used for further analysis or anomaly detection.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name        value                                      component_KPI            timestamp_utc8
840734  1615278600   Mysql02                        Mysql-MySQL_3306_Bytes Sent  110722.8525                Mysql02-Mysql-MySQL_3306_Bytes Sent 2021-03-09 16:30:00+08:00
840735  1615278600   Mysql02                    Mysql-MySQL_3306_Bytes Received -246528.9016            Mysql02-Mysql-MySQL_3306_Bytes Received 2021-03-09 16:30:00+08:00
840736  1615278600   Mysql02             Mysql-MySQL_3306_Binlog stmt cache use    1441.0000     Mysql02-Mysql-MySQL_3306_Binlog stmt cache use 2021-03-09 16:30:00+08:00
840737  1615278600   Mysql02        Mysql-MySQL_3306_Binlog stmt cache disk use       0.0000  Mysql02-Mysql-MySQL_3306_Binlog stmt cache dis... 2021-03-09 16:30:00+08:00
840738  1615278600  dockerA2  Container-DOCKER_CONTAINER_cd2b4a29291e--bcou-...       0.0000  dockerA2-Container-DOCKER_CONTAINER_cd2b4a2929... 2021-03-09 16:30:00+08:00
...            ...       ...                                                ...          ...                                                ...                       ...
866668  1615280340   Mysql02                       Mysql-MySQL_3306_Open Tables    2475.0000               Mysql02-Mysql-MySQL_3306_Open Tables 2021-03-09 16:59:00+08:00
866669  1615280340   Mysql02                        Mysql-MySQL_3306_Open Files     207.0000                Mysql02-Mysql-MySQL_3306_Open Files 2021-03-09 16:59:00+08:00
866670  1615280340   Mysql01              Mysql-MySQL_3306_LongestTrxActiveTime       0.0000      Mysql01-Mysql-MySQL_3306_LongestTrxActiveTime 2021-03-09 16:59:00+08:00
866671  1615280340   Mysql01              Mysql-MySQL_3306_Max Used Connections     108.0000      Mysql01-Mysql-MySQL_3306_Max Used Connections 2021-03-09 16:59:00+08:00
866672  1615280340   Mysql01            Mysql-MySQL_3306_Slave Open Temp Tables       0.0000    Mysql01-Mysql-MySQL_3306_Slave Open Temp Tables 2021-03-09 16:59:00+08:00

[25939 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered data with the P95 thresholds to perform anomaly detection
filtered_data_with_thresholds = filtered_data.merge(p95_thresholds, on='component_KPI', how='left')

# Identify anomalies where the value exceeds the P95 threshold
filtered_data_with_thresholds['is_anomaly'] = filtered_data_with_thresholds['value'] > filtered_data_with_thresholds['P95_threshold']

# Extract only the anomalies for further analysis
anomalies = filtered_data_with_thresholds[filtered_data_with_thresholds['is_anomaly']]

anomalies

```
Out[4]:
```
Anomaly detection has been successfully performed on the filtered dataset for March 9, 2021, from 16:30 to 17:00 (UTC+8). A total of 490 data points were identified as anomalies, where the values exceeded their respective global P95 thresholds. These anomalies span various 'component-KPI' combinations and are marked for further analysis to investigate potential issues in the system.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  cmdb_id                                          kpi_name      value                                      component_KPI            timestamp_utc8  P95_threshold  is_anomaly
9      1615278600  Mysql02                 Mysql-MySQL_3306_Binlog Cache Use   216.9016          Mysql02-Mysql-MySQL_3306_Binlog Cache Use 2021-03-09 16:30:00+08:00     185.379675        True
13     1615278600  Mysql02            Mysql-MySQL_3306_Innodb Row Lock Waits     1.6066     Mysql02-Mysql-MySQL_3306_Innodb Row Lock Waits 2021-03-09 16:30:00+08:00       0.733300        True
14     1615278600  Mysql02             Mysql-MySQL_3306_Innodb Row Lock Time    12.4098      Mysql02-Mysql-MySQL_3306_Innodb Row Lock Time 2021-03-09 16:30:00+08:00      10.251050        True
16     1615278600  Mysql02                   Mysql-MySQL_3306_Handler Update    17.7377            Mysql02-Mysql-MySQL_3306_Handler Update 2021-03-09 16:30:00+08:00      15.548550        True
24     1615278600  Mysql02                 Mysql-MySQL_3306_Handler Read Key   138.2787          Mysql02-Mysql-MySQL_3306_Handler Read Key 2021-03-09 16:30:00+08:00     123.464800        True
...           ...      ...                                               ...        ...                                                ...                       ...            ...         ...
25833  1615280340  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite   749.0667  Mysql01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sd... 2021-03-09 16:59:00+08:00     735.226680        True
25845  1615280340  Mysql01                        Mysql-MySQL_3306_Sort Scan     4.8033                 Mysql01-Mysql-MySQL_3306_Sort Scan 2021-03-09 16:59:00+08:00       3.539980        True
25858  1615280340  Mysql02            Mysql-MySQL_3306_Table Locks Immediate     0.0667     Mysql02-Mysql-MySQL_3306_Table Locks Immediate 2021-03-09 16:59:00+08:00       0.050800        True
25898  1615280340  Mysql01                      Mysql-MySQL_3306_Select Scan     4.9180               Mysql01-Mysql-MySQL_3306_Select Scan 2021-03-09 16:59:00+08:00       3.676680        True
25911  1615280340  Mysql01                        Mysql-MySQL_3306_Rows Read  2597.3607                 Mysql01-Mysql-MySQL_3306_Rows Read 2021-03-09 16:59:00+08:00    1939.903320        True

[490 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Group anomalies by 'component-KPI' and sort by timestamp to identify consecutive anomalies
anomalies = anomalies.sort_values(by=['component_KPI', 'timestamp_utc8'])
anomalies['time_diff'] = anomalies.groupby('component_KPI')['timestamp_utc8'].diff().dt.total_seconds()

# Define a threshold for consecutive anomalies (e.g., anomalies within 5 minutes are considered part of the same fault)
consecutive_threshold = 300  # 5 minutes in seconds
anomalies['fault_group'] = (anomalies['time_diff'] > consecutive_threshold).cumsum()

# Group consecutive anomalies into faults
faults = anomalies.groupby(['component_KPI', 'fault_group']).agg(
    start_time=('timestamp_utc8', 'min'),
    end_time=('timestamp_utc8', 'max'),
    max_value=('value', 'max'),
    threshold=('P95_threshold', 'first')
).reset_index()

# Calculate the threshold breach percentage
faults['threshold_breach_percentage'] = (faults['max_value'] - faults['threshold']) / faults['threshold'] * 100

# Filter out isolated noise spikes and faults with threshold breach <= 50%
filtered_faults = faults[faults['threshold_breach_percentage'] > 50]

filtered_faults

```
Out[5]:
```
Fault identification has been successfully performed by grouping consecutive anomalies for each 'component-KPI' time series in the filtered dataset. A total of 68 faults were identified, where the threshold breach exceeded 50% of the extremal value. These faults represent significant deviations from normal behavior and are suitable for further investigation to diagnose potential issues in the system.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI  fault_group                start_time                  end_time  max_value  threshold  threshold_breach_percentage
14   IG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...            5 2021-03-09 16:40:00+08:00 2021-03-09 16:40:00+08:00   8.000000   4.000000                   100.000000
15   IG01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...            5 2021-03-09 16:50:00+08:00 2021-03-09 16:50:00+08:00   1.900000   0.200000                   850.000000
18   IG01-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE...            5 2021-03-09 16:42:00+08:00 2021-03-09 16:42:00+08:00   1.000000   0.000000                          inf
29                         IG02-OSLinux-CPU_CPU_CPUWio            9 2021-03-09 16:30:00+08:00 2021-03-09 16:30:00+08:00   2.735398   0.680538                   301.946490
31   IG02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_D...           10 2021-03-09 16:42:00+08:00 2021-03-09 16:42:00+08:00   5.000000   2.000000                   150.000000
..                                                 ...          ...                       ...                       ...        ...        ...                          ...
333                    apache01-OSLinux-CPU_CPU_CPUWio          106 2021-03-09 16:58:00+08:00 2021-03-09 16:58:00+08:00   0.531700   0.311342                    70.776831
337           apache02-OSLinux-CPU_CPU-0_SingleCpuUtil          107 2021-03-09 16:59:00+08:00 2021-03-09 16:59:00+08:00   1.531000   0.997330                    53.509871
341                   apache02-OSLinux-CPU_CPU_CPULoad          107 2021-03-09 16:35:00+08:00 2021-03-09 16:35:00+08:00   0.210000   0.106000                    98.113208
347  apache02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...          109 2021-03-09 16:39:00+08:00 2021-03-09 16:43:00+08:00   5.000000   2.900000                    72.413793
348  apache02-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...          109 2021-03-09 16:48:00+08:00 2021-03-09 16:48:00+08:00   4.000000   0.000000                          inf

[68 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Identify the fault with the highest threshold breach percentage
most_significant_fault = filtered_faults.loc[filtered_faults['threshold_breach_percentage'].idxmax()]

# Determine the level of the component associated with this fault
component_id = most_significant_fault['component_KPI'].split('-')[0]
if component_id.startswith('docker'):
    component_level = 'container'
elif component_id.startswith('apache') or component_id.startswith('Tomcat') or component_id.startswith('Mysql') or component_id.startswith('Redis'):
    component_level = 'service'
else:
    component_level = 'node'

most_significant_fault, component_level

```
Out[6]:
```
The fault with the most significant deviation from the threshold is associated with the 'IG01-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE_WAIT' component-KPI. This fault occurred on March 9, 2021, at 16:42 (UTC+8) and is linked to the node-level component 'IG01'. The threshold breach percentage is infinite, indicating an extreme deviation from the expected behavior.

The original code execution output of IPython Kernel is also provided below for reference:

(component_KPI                  IG01-OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE...
fault_group                                                                    5
start_time                                             2021-03-09 16:42:00+08:00
end_time                                               2021-03-09 16:42:00+08:00
max_value                                                                    1.0
threshold                                                                    0.0
threshold_breach_percentage                                                  inf
Name: 18, dtype: object, 'node')```
```